# Thể hiện returns data đối với các loại mặt hàng (cả rate và total)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. ĐỌC DỮ LIỆU
# ==========================================
df_returns = pd.read_csv('../dataset/returns.csv')
df_products = pd.read_csv('../dataset/products.csv')
df_customers = pd.read_csv('../dataset/customers.csv')

# Cần order_items để tính Return Rate (so sánh lượng trả / lượng bán)
df_order_items = pd.read_csv('../dataset/order_items.csv')

# Cần orders để lấy customer_id cho mỗi order_id bị return
df_orders = pd.read_csv('../dataset/orders.csv')

# ==========================================
# 2. XỬ LÝ DỮ LIỆU
# ==========================================
# Gộp Category từ bảng Products vào Returns và Order Items
df_ret_prod = pd.merge(df_returns, df_products[['product_id', 'category']], on='product_id', how='left')
df_oi_prod = pd.merge(df_order_items, df_products[['product_id', 'category']], on='product_id', how='left')

# --- Phân tích 1: Loại mặt hàng bị return nhiều nhất (Rate và Total) ---
# Lượng bán ra theo category
cat_sold = df_oi_prod.groupby('category')['quantity'].sum().reset_index(name='total_sold')
# Lượng trả về theo category
cat_ret = df_ret_prod.groupby('category')['return_quantity'].sum().reset_index(name='total_returned')

cat_stats = pd.merge(cat_ret, cat_sold, on='category', how='left').fillna(0)
cat_stats['return_rate_pct'] = (cat_stats['total_returned'] / cat_stats['total_sold']) * 100

# --- Phân tích 2 & 3: Đối tượng return nhiều nhất & Mặt hàng theo đối tượng ---
# Kết nối Returns -> Orders -> Customers
df_ret_cust = pd.merge(df_ret_prod, df_orders[['order_id', 'customer_id']], on='order_id', how='left')
df_ret_full = pd.merge(df_ret_cust, df_customers[['customer_id', 'age_group', 'gender']], on='customer_id', how='left')

# Loại bỏ các dòng thiếu thông tin age_group (nếu có)
df_ret_full = df_ret_full.dropna(subset=['age_group', 'category'])

# Ma trận: Độ tuổi vs Loại mặt hàng
pivot_age_cat = df_ret_full.pivot_table(index='age_group', columns='category', values='return_quantity', aggfunc='sum', fill_value=0)

# Tổng trả hàng theo Đối tượng (Độ tuổi & Giới tính)
demo_ret = df_ret_full.groupby(['age_group', 'gender'])['return_quantity'].sum().reset_index()


# ==========================================
# 3. TRỰC QUAN HÓA (VISUALIZATION)
# ==========================================
sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(18, 18))

# Chia lưới hiển thị cho các biểu đồ
gs = fig.add_gridspec(3, 2)

# --- Biểu đồ 1a: Tổng số lượng hàng bị Return theo Category ---
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(data=cat_stats.sort_values('total_returned', ascending=False), 
            x='total_returned', y='category', palette='Reds_r', ax=ax1)
ax1.set_title('Tổng Số Lượng Bị Return Theo Mặt Hàng (Total Returns)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Số lượng trả về', fontsize=12)
ax1.set_ylabel('Danh mục (Category)', fontsize=12)

# --- Biểu đồ 1b: Tỷ lệ Return theo Category (Return Rate %) ---
ax2 = fig.add_subplot(gs[0, 1])
sns.barplot(data=cat_stats.sort_values('return_rate_pct', ascending=False), 
            x='return_rate_pct', y='category', palette='Oranges_r', ax=ax2)
ax2.set_title('Tỷ Lệ Bị Return Theo Mặt Hàng (Return Rate %)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Tỷ lệ Return (%)', fontsize=12)
ax2.set_ylabel('')

# --- Biểu đồ 2: Mặt hàng được return nhiều nhất theo Đối tượng (Heatmap) ---
ax3 = fig.add_subplot(gs[1, :])
sns.heatmap(pivot_age_cat, cmap='YlOrRd', annot=True, fmt='g', linewidths=.5, ax=ax3)
ax3.set_title('Sự Phân Bổ Số Lượng Return Giữa Độ Tuổi (Age Group) Và Mặt Hàng', fontsize=14, fontweight='bold')
ax3.set_xlabel('Danh mục (Category)', fontsize=12)
ax3.set_ylabel('Nhóm độ tuổi', fontsize=12)

# --- Biểu đồ 3: Đối tượng nào return nhiều nhất ---
ax4 = fig.add_subplot(gs[2, :])
sns.barplot(data=demo_ret, x='age_group', y='return_quantity', hue='gender', palette='Set2', ax=ax4)
ax4.set_title('Đối Tượng (Độ Tuổi & Giới Tính) Có Lượng Hoàn Trả Nhiều Nhất', fontsize=14, fontweight='bold')
ax4.set_xlabel('Nhóm độ tuổi', fontsize=12)
ax4.set_ylabel('Tổng số lượng trả về', fontsize=12)
ax4.legend(title='Giới tính (Gender)', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

# Tương quan trả hàng và web visit

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. ĐỌC VÀ CHUẨN BỊ DỮ LIỆU
# ==========================================
df_returns = pd.read_csv('../dataset/returns.csv')
df_traffic = pd.read_csv('../dataset/web_traffic.csv')
df_order_items = pd.read_csv('../dataset/order_items.csv')

# Chuyển đổi định dạng ngày
df_returns['return_date'] = pd.to_datetime(df_returns['return_date'])
df_traffic['date'] = pd.to_datetime(df_traffic['date'])

# Tạo cột Tháng để gom nhóm tính tính mùa vụ (Seasonality)
df_returns['month'] = df_returns['return_date'].dt.month
df_traffic['month'] = df_traffic['date'].dt.month

# ==========================================
# 2. TÍNH TOÁN CÁC CHỈ SỐ THEO THÁNG
# ==========================================

# 2.1. Tính Tỷ lệ trả hàng trung bình (Monthly Return Rate)
# Ta cần tổng lượng bán từ order_items (giả định có cột ngày hoặc map qua bảng orders)
# Ở đây ta tính dựa trên lượng trả hàng trung bình mỗi tháng để xem biến động
monthly_returns = df_returns.groupby('month')['return_quantity'].mean().reset_index()

# 2.2. Tính Traffic trung bình mỗi tháng
monthly_traffic = df_traffic.groupby('month').agg({
    'sessions': 'mean',
    'unique_visitors': 'mean',
    'bounce_rate': 'mean'
}).reset_index()

# ==========================================
# 3. TRỰC QUAN HÓA SỰ TƯƠNG QUAN
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(16, 16))

# --- BIỂU ĐỒ 1: TƯƠNG QUAN GIỮA TRAFFIC VÀ LƯỢNG TRẢ HÀNG ---
ax1 = axes[0]
ax1_twin = ax1.twinx()

# Vẽ lượt truy cập (Sessions)
sns.lineplot(data=monthly_traffic, x='month', y='sessions', marker='o', color='#1f77b4', linewidth=3, label='Lượt truy cập (Sessions)', ax=ax1)
# Vẽ lượng trả hàng (Returns)
sns.barplot(data=monthly_returns, x='month', y='return_quantity', color='#d62728', alpha=0.4, label='Lượng trả hàng trung bình', ax=ax1_twin)

ax1.set_title('EDA: SỰ TƯƠNG QUAN GIỮA LƯỢT TRUY CẬP VÀ HOÀN TRẢ HÀNG', fontsize=15, fontweight='bold')
ax1.set_xlabel('Tháng trong năm')
ax1.set_ylabel('Số lượt Sessions')
ax1_twin.set_ylabel('Số lượng hàng trả (Returns)')
ax1.set_xticks(range(1, 13))

# --- BIỂU ĐỒ 2: CHẤT LƯỢNG TRAFFIC (BOUNCE RATE) ---
ax2 = axes[1]
sns.lineplot(data=monthly_traffic, x='month', y='bounce_rate', marker='s', color='#9467bd', linewidth=2.5, ax=ax2)

ax2.set_title('EDA: TỶ LỆ THOÁT TRANG (BOUNCE RATE) THEO THÁNG', fontsize=15, fontweight='bold')
ax2.set_ylabel('Bounce Rate')
ax2.set_xlabel('Tháng trong năm')
ax2.set_xticks(range(1, 13))

plt.tight_layout()
plt.show()

# --- XUẤT CÁC "KEYS" PHÂN TÍCH ---
print("--- CHÌA KHÓA PHÂN TÍCH TRAFFIC & RETURNS ---")
peak_traffic_month = monthly_traffic.loc[monthly_traffic['sessions'].idxmax(), 'month']
lowest_bounce_month = monthly_traffic.loc[monthly_traffic['bounce_rate'].idxmin(), 'month']

print(f"Key 9 (Đỉnh Traffic): Tháng {int(peak_traffic_month)} - Kiểm tra xem có trùng với tháng Promo không?")
print(f"Key 10 (Traffic chất lượng nhất): Tháng {int(lowest_bounce_month)} - Tỷ lệ thoát thấp nhất.")